# 02 — Exploración de Datasets BOP: T-LESS y YCB-Video

**Ejecutable localmente en M1 Pro**

Exploramos:
1. Estructura y estadísticas de cada dataset
2. Visualización de samples RGB-D
3. Modelos 3D con Open3D
4. Distribución de objetos y poses GT
5. Comparación de propiedades (simetría, textura, etc.)

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.utils.dataset_loader import BOPDataset, verify_dataset
from src.utils.visualization import draw_pose_axes, draw_projected_points
from src.utils.lie_groups import pose_to_Rt, geodesic_distance_SO3
import cv2

DATA_DIR = Path('../data/datasets')
print('Data dir:', DATA_DIR.resolve())

## 1. Verificar Integridad de Datasets

In [ ]:
for ds_name in ['tless', 'ycbv']:
    ds_path = DATA_DIR / ds_name
    if ds_path.exists():
        verify_dataset(str(ds_path), 'test')
    else:
        print(f'{ds_name}: NOT FOUND')

## 2. Explorar T-LESS

In [ ]:
tless = BOPDataset(str(DATA_DIR / 'tless'), split='test')
print(tless)
print(f'\nObjects: {tless.get_object_ids()}')
print(f'Scenes: {tless.get_scene_ids()}')

# Object properties
print('\nObject Diameters (mm):')
for obj_id in tless.get_object_ids()[:10]:
    d = tless.get_object_diameter(obj_id)
    sym = tless.get_symmetries(obj_id)
    n_sym = len(sym.get('symmetries_discrete', [])) + len(sym.get('symmetries_continuous', []))
    print(f'  Obj {obj_id:02d}: diameter={d:.1f}mm, symmetries={n_sym}')

In [ ]:
# Visualizar samples de T-LESS
scenes = tless.get_scene_ids()
if scenes:
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    
    for idx, scene_id in enumerate(scenes[:4]):
        try:
            sample = tless.load_sample(scene_id, 0)
            
            # RGB
            axes[0, idx].imshow(sample['rgb'])
            axes[0, idx].set_title(f'Scene {scene_id} — RGB\n{len(sample["gt_poses"])} objects')
            axes[0, idx].axis('off')
            
            # Depth
            depth_vis = sample['depth'].astype(float)
            depth_vis[depth_vis == 0] = np.nan
            axes[1, idx].imshow(depth_vis, cmap='turbo')
            axes[1, idx].set_title(f'Scene {scene_id} — Depth')
            axes[1, idx].axis('off')
        except Exception as e:
            axes[0, idx].text(0.5, 0.5, f'Error: {e}', ha='center', transform=axes[0, idx].transAxes)
            axes[0, idx].axis('off')
            axes[1, idx].axis('off')
    
    plt.suptitle('T-LESS Test Samples (industrial objects, textureless)', fontsize=14)
    plt.tight_layout()
    plt.savefig('../experiments/tless_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No test scenes. Extract tless_test_primesense_all.zip first.')

## 3. Explorar YCB-Video

In [ ]:
ycbv = BOPDataset(str(DATA_DIR / 'ycbv'), split='test')
print(ycbv)
print(f'\nObjects: {ycbv.get_object_ids()}')
print(f'Scenes: {ycbv.get_scene_ids()}')

print('\nObject Diameters (mm):')
for obj_id in ycbv.get_object_ids()[:10]:
    d = ycbv.get_object_diameter(obj_id)
    print(f'  Obj {obj_id:02d}: diameter={d:.1f}mm')

In [ ]:
# Visualizar samples de YCB-Video
scenes = ycbv.get_scene_ids()
if scenes:
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    
    for idx, scene_id in enumerate(scenes[:4]):
        try:
            sample = ycbv.load_sample(scene_id, 1)
            
            axes[0, idx].imshow(sample['rgb'])
            axes[0, idx].set_title(f'Scene {scene_id} — RGB\n{len(sample["gt_poses"])} objects')
            axes[0, idx].axis('off')
            
            depth_vis = sample['depth'].astype(float)
            depth_vis[depth_vis == 0] = np.nan
            axes[1, idx].imshow(depth_vis, cmap='turbo')
            axes[1, idx].set_title(f'Scene {scene_id} — Depth')
            axes[1, idx].axis('off')
        except Exception as e:
            axes[0, idx].text(0.5, 0.5, f'Error: {e}', ha='center', transform=axes[0, idx].transAxes)
            axes[0, idx].axis('off')
            axes[1, idx].axis('off')
    
    plt.suptitle('YCB-Video Test Samples (household objects)', fontsize=14)
    plt.tight_layout()
    plt.savefig('../experiments/ycbv_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No test scenes. Extract ycbv_test_all.zip first.')

## 4. Visualizar Modelos 3D

In [ ]:
import trimesh

# Cargar modelos YCB-Video (los primeros 6)
fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': '3d'})

for idx, obj_id in enumerate(ycbv.get_object_ids()[:6]):
    row, col = idx // 3, idx % 3
    model_path = ycbv.get_model_path(obj_id)
    if model_path.exists():
        mesh = trimesh.load(str(model_path))
        vertices = np.array(mesh.vertices)
        
        # Subsample for visualization
        if len(vertices) > 2000:
            idx_sub = np.random.choice(len(vertices), 2000, replace=False)
            vertices = vertices[idx_sub]
        
        ax = axes[row, col]
        ax.scatter(vertices[:, 0], vertices[:, 1], vertices[:, 2],
                  s=0.5, c=vertices[:, 2], cmap='viridis', alpha=0.6)
        d = ycbv.get_object_diameter(obj_id)
        ax.set_title(f'Obj {obj_id} (d={d:.0f}mm)')
    else:
        axes[row, col].text(0.5, 0.5, 'Not found', ha='center')

plt.suptitle('YCB-Video 3D Models (point clouds)', fontsize=14)
plt.tight_layout()
plt.savefig('../experiments/ycbv_models_3d.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Visualizar Poses GT con Ejes 3D

In [ ]:
# Tomar un sample y dibujar ejes de pose
datasets = {'T-LESS': tless, 'YCB-Video': ycbv}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (name, ds) in enumerate(datasets.items()):
    scenes = ds.get_scene_ids()
    if not scenes:
        axes[idx].text(0.5, 0.5, f'{name}: no test images yet', ha='center',
                       transform=axes[idx].transAxes)
        axes[idx].axis('off')
        continue
    
    try:
        sample = ds.load_sample(scenes[0], 0)
        img_bgr = cv2.cvtColor(sample['rgb'], cv2.COLOR_RGB2BGR)
        K = sample['cam_K']
        
        # Draw axes for each object
        for gt in sample['gt_poses']:
            R = gt['cam_R_m2c']
            t = gt['cam_t_m2c'] / 1000.0  # mm → m
            img_bgr = draw_pose_axes(img_bgr, R, t, K, axis_length=0.03)
        
        axes[idx].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
        axes[idx].set_title(f'{name} — GT Pose Axes ({len(sample["gt_poses"])} objects)')
    except Exception as e:
        axes[idx].text(0.5, 0.5, f'Error: {e}', ha='center',
                       transform=axes[idx].transAxes)
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig('../experiments/gt_pose_axes.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Comparativa de Datasets

| Propiedad | T-LESS | YCB-Video |
|-----------|--------|----------|
| Tipo objetos | Industriales, sin textura | Domésticos, con textura |
| Nº objetos | 30 | 21 |
| Simetrías | Muchas (piezas industriales) | Pocas |
| Desafío principal | Falta de textura, simetrías | Oclusiones, clutter |
| Sensor | Primesense Carmine 1.09 | Asus Xtion PRO |
| Relevancia TFM | Bin picking industrial | Benchmark general |